# 03 - Key plots and figures

Load the `.npy` statistics written by `evaluate.py` and reproduce the main
benchmark figures: merged accuracy and reconstructed-photon fraction, scanned
vs photon count (single flash) and vs inter-flash gap Δt (double flash).

In [1]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = '../results'   # output_dir from configs/evaluation.yaml

## Single-flash: metrics vs photon count
`single_flash.npy` is `{model_name: {merge_acc, merge_pure, interval, reco_frac}}`,
each a list indexed by increasing photon count.

In [2]:
single = np.load(f'{RESULTS_DIR}/single_flash.npy', allow_pickle=True).item()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, stats in single.items():
    ax[0].plot(stats['merge_acc'], label=name)
    ax[1].plot(stats['reco_frac'], label=name)
ax[0].set_title('merged accuracy'); ax[0].set_xlabel('photon-count bin')
ax[1].set_title('reconstructed photon fraction'); ax[1].set_xlabel('photon-count bin')
ax[1].axhline(1.0, color='k', ls='--', lw=1)
for a in ax: a.legend(); a.grid(True, alpha=0.3)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '../results/single_flash.npy'

## Double-flash: per-flash metrics vs Δt
`double_flash.npy` is `{model_name: {merge_acc_flash1/2, reco_frac_flash1/2, merge_pure, bin_counts}}`,
each a length-`max_dt` array already averaged per Δt. We rebin for a smoother curve.

In [ ]:
double = np.load(f'{RESULTS_DIR}/double_flash.npy', allow_pickle=True).item()

def rebin(arr, bin_size=30):
    arr = np.asarray(arr)
    n = (len(arr) // bin_size) * bin_size
    return arr[:n].reshape(-1, bin_size).mean(axis=1)

fig, ax = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for name, stats in double.items():
    dt = rebin(np.arange(len(stats['merge_acc_flash1'])))
    ax[0, 0].plot(dt, rebin(stats['merge_acc_flash1']), label=name)
    ax[0, 1].plot(dt, rebin(stats['merge_acc_flash2']), label=name)
    ax[1, 0].plot(dt, rebin(stats['reco_frac_flash1']), label=name)
    ax[1, 1].plot(dt, rebin(stats['reco_frac_flash2']), label=name)
ax[0, 0].set_title('merged accuracy - flash 1')
ax[0, 1].set_title('merged accuracy - flash 2')
ax[1, 0].set_title('reco photon fraction - flash 1')
ax[1, 1].set_title('reco photon fraction - flash 2')
for a in ax.flat: a.legend(); a.grid(True, alpha=0.3); a.set_xlabel('Δt (bins)')
plt.tight_layout(); plt.show()

## Visualize one prediction
Overlay a model's per-bin flash probability on a waveform from any dataset.

In [ ]:
import torch
from flashdet.data import make_dataloader
from flashdet.utils import load_models

device = 'cuda' if torch.cuda.is_available() else 'cpu'
models = load_models('../configs/evaluation.yaml', device=device)  # requires checkpoints
name, (model, _) = next(iter(models.items()))

loader = make_dataloader('demo_dataset.npy', batch_size=4, shuffle=False)
data, _, hit_times, _, _ = next(iter(loader))
with torch.no_grad():
    class_logits, _ = model(data.to(device), mode='bce')
prob = torch.sigmoid(class_logits)[0, 0].cpu().numpy()

fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(data[0, 0].numpy(), lw=1); ax[0].set_ylabel('ADC')
ax[1].plot(prob, lw=1, color='C1'); ax[1].set_ylabel('p(flash)')
for t in hit_times[0]:
    if t > 0:
        ax[0].axvline(int(t), color='r', ls='--', lw=1)
        ax[1].axvline(int(t), color='r', ls='--', lw=1)
ax[0].set_title(f'{name} prediction'); ax[1].set_xlabel('time bin')
plt.show()